# 01 — Setup, Validation & Exploratory Data Analysis

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

This notebook does four things, in order:

1. loads all eight datasets through the validated loader,
2. runs the automated leakage scanners and prints the resulting register,
3. produces the exploratory tables and figures that feed the D7 guide and the D8 deck,
4. builds and *audits* the cross-validation scheme for every module.

> **It contains no pipeline logic.** Every table comes from `novafin.data.profile`,
> every figure from `novafin.viz`, every check from `novafin.data.validate`. If you
> find yourself writing a loop with business meaning in a cell here, it belongs in a
> module — that is what makes the results reproducible and testable.

**Prerequisite:** run `00_environment_check.ipynb` first.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/ePGD - MLDS IIT Bombay/C5 ML In Finanace/Data"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin.config import load_config
from novafin.data import (
    check_entity_overlap, class_balance, column_profile, correlation_with_target,
    dataset_overview, decile_lift, describe_splits, group_rate, holdout_by_time,
    load_all, load_dataset, make_feature_frame, make_splitter, numeric_summary,
    temporal_rate, validate_all,
)
from novafin.data.validate import combined_report_frame
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, save_figure
from novafin import viz

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "01_eda.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
print("config fingerprint:", cfg.fingerprint())

## 2 · Load every dataset

`load_dataset` is the only sanctioned way into the data. On every call it

* asserts the row count, column count and positive rate against the **Phase-0 audit
  values stored in `configs/config.yaml`** — so if the file on disk is not the file
  that was audited, you find out immediately rather than in the results section;
* records the file's **SHA-256**, which travels into the run manifest;
* downcasts numerics (the order book drops from ~64 MB to ~45 MB);
* sorts causally by the time key, so a later `shift` or `rolling` is meaningful.

In [ ]:
results = load_all(cfg=cfg)
overview = dataset_overview(results)
overview

In [ ]:
frames = {key: result.frame for key, result in results.items()}

print(f"Total in memory: {overview['memory_mb'].sum():.1f} MB")
print(f"Colab free-tier RAM: ~12,000 MB  ->  headroom is not a concern")
print(f"All schemas match the Phase-0 audit: {overview['schema_ok'].all()}")

## 3 · Automated leakage detection

This is the heart of the project's validation story. `validate_dataset` does **not**
read `docs/LEAKAGE_REGISTER.md` — it rediscovers the findings from the data using four
families of detector:

| Detector | Catches |
|---|---|
| `detect_identity_columns` | a column that is an exact sum/difference of two others |
| `detect_target_aliases` | a column whose linear correlation with the target exceeds 0.95 |
| `detect_label_reconstruction` | a continuous column that **is** a categorical label under thresholds |
| `detect_same_row_derivation` | a target that is the same-row percentage change of a price |

A finding is **CRITICAL** only when it is *undeclared*. Anything already in
`drop_always` (removed from X automatically) or `acknowledged_leaks` (retained on
purpose, controlled per experiment) is reported as INFO with its register ID, because
the register is the record of decisions already taken.

In [ ]:
reports = validate_all(frames, cfg=cfg)

for key, report in reports.items():
    status = "PASS" if report.ok else f"FAIL ({len(report.critical)} critical)"
    print(f"{key:<14} {status}")

print()
print("Any undeclared CRITICAL leak anywhere:",
      any(not r.ok for r in reports.values()))

In [ ]:
# The findings that matter, in register order.
interesting = {
    "arithmetic_identity", "target_alias", "label_reconstruction",
    "same_row_derivation", "signal_strength", "cv_power", "near_constant",
}
rows = combined_report_frame(list(reports.values()))
rows[rows["check"].isin(interesting)].reset_index(drop=True)

### 3.1 · Signal strength per module — the chart that frames everything

The dashed lines sit at the **alias threshold of 0.95**. A healthy module's strongest
bar is far below it. Reading across the eight modules:

* a maximum near **1.0** means a leak,
* a maximum around **0.1 – 0.5** is a normal, learnable problem,
* a maximum near **0.0** is a null result — which is what the churn module shows, and
  which we report honestly rather than tune away.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, (key, frame) in zip(axes.ravel(), frames.items(), strict=False):
    spec = cfg.dataset(key)
    admissible = frame.drop(
        columns=[c for c in spec.forbidden_features if c in frame.columns],
        errors="ignore",
    )
    admissible[spec.target] = frame[spec.target]
    corr = correlation_with_target(admissible, spec.target, top=6)
    if not len(corr):
        ax.axis("off")
        continue
    values = corr.sort_values(key=np.abs)
    # Acknowledged leaks (retained on purpose) are drawn in the risk colour
    # so a near-1.0 bar is never mistaken for a legitimate feature.
    ack = set(spec.acknowledged_leaks or [])
    colors = ["#B88F20" if c in ack else "#00B2A9" for c in values.index]
    ax.barh(values.index, values.to_numpy(), color=colors)
    ax.axvline(0.95, ls="--", lw=0.9, color="#B88F20")
    ax.axvline(-0.95, ls="--", lw=0.9, color="#B88F20")
    ax.set_xlim(-1.05, 1.05)
    ax.set_title(f"{key}\n|r|max = {values.abs().max():.3f}", fontsize=10)
    ax.tick_params(labelsize=7)
fig.suptitle("Correlation with target among ADMISSIBLE features (dashed = leakage threshold)")
fig.tight_layout()
save_figure(fig, "01_signal_strength_by_module", close=False)
plt.show()

### 3.2 · L-01 — the market target is the *same-day* return

The single most consequential finding in the whole project, and the one the brief's
own wording ("predict future returns") walks you straight into.

In [ ]:
market = frames["market"].copy()
market["same_day"] = market.groupby("Ticker", observed=True)["Close"].pct_change()
market["next_day"] = market.groupby("Ticker", observed=True)["Return"].shift(-1)
mask = market[["Return", "same_day", "next_day"]].notna().all(axis=1)

print(f"corr(Return, SAME-day close-to-close) = {market.loc[mask, 'Return'].corr(market.loc[mask, 'same_day']):+.4f}")
print(f"corr(Return, NEXT-day return)         = {market.loc[mask, 'Return'].corr(market.loc[mask, 'next_day']):+.4f}")
print()
print("Verified SAFE (trailing, not forward-looking):")
print(f"  corr(Momentum_20D,   trailing 20d return) = "
      f"{market['Momentum_20D'].corr(market.groupby('Ticker', observed=True)['Close'].pct_change(20)):+.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(market.loc[mask, "same_day"], market.loc[mask, "Return"], s=2, alpha=0.3, color="#B88F20")
ax[0].set_title("LEAKED: Return vs same-day return")
ax[0].set_xlabel("Same-day close-to-close"); ax[0].set_ylabel("Return")
ax[1].scatter(market.loc[mask, "next_day"], market.loc[mask, "Return"], s=2, alpha=0.3, color="#00B2A9")
ax[1].set_title("CLEAN: Return vs next-day return")
ax[1].set_xlabel("Next-day return"); ax[1].set_ylabel("Return")
fig.tight_layout()
save_figure(fig, "01_L01_market_return_leakage", close=False)
plt.show()

### 3.3 · L-03 — the liquidity gap is arithmetic, not a forecast

In [ ]:
liq = frames["liquidity"]
residual = (liq["Liquidity_Gap"] - (liq["Expected_Outflows"] - liq["Expected_Inflows"])).abs()
print(f"max |Liquidity_Gap - (Outflows - Inflows)| = {residual.max():.5f}")
print(f"Liquidity_Buffer: mean {liq['Liquidity_Buffer'].mean():,.1f}, "
      f"std {liq['Liquidity_Buffer'].std():.2f}, "
      f"CV {liq['Liquidity_Buffer'].std() / liq['Liquidity_Buffer'].mean():.5f}")
print("-> the gap is derived, and the buffer has no variance to model.")
print("   M7 therefore forecasts Expected_Outflows h days ahead and DERIVES the gap.")

### 3.4 · L-02 — three columns, one answer

In [ ]:
from novafin.data.validate import detect_label_reconstruction

hft = frames["hft"]
agreement = detect_label_reconstruction(hft, "Price_Move_Class", "Future_Return_100ms")
print(f"thresholds on Future_Return_100ms reproduce {agreement:.1%} of Price_Move_Class labels")
print()
print("Genuine microstructure signal (these are the features we KEEP):")
ordinal = pd.Series(pd.Categorical(hft["Price_Move_Class"],
                                   categories=["DOWN", "FLAT", "UP"], ordered=True).codes)
for column in ["OBI_3Level", "OBI_Level1", "Microprice_Minus_Mid", "Relative_Spread",
               "Cancellation_Rate", "Total_Depth"]:
    print(f"  corr({column:<22}, Price_Move_Class) = {hft[column].corr(ordinal):+.4f}")

## 4 · Per-module exploratory analysis

From here each module gets the same four artefacts, so the eight are comparable:
a column profile, a class-balance view, a decile-lift table for the strongest feature,
and a drift check where a time key exists.

### 4.1 · M2 Credit Risk — `nova_loans.csv`

In [ ]:
loans = frames["loans"]
column_profile(loans)

In [ ]:
balance = class_balance(loans, "Default_Flag")
display(balance)

fig = viz.plot_class_balance(balance, "Default_Flag")
save_figure(fig, "01_m02_class_balance", close=False); plt.show()

lift = decile_lift(loans, "Interest_Rate", "Default_Flag")
display(lift)
fig = viz.plot_decile_lift(lift, "Interest_Rate", "Default_Flag")
save_figure(fig, "01_m02_interest_rate_lift", close=False); plt.show()

**L-06 — the `Interest_Rate` judgement call.**

`Interest_Rate` is the strongest single feature. It is *available* at scoring time, so
it is not strictly leakage — but under risk-based pricing it partly encodes NovaFin's
existing scorecard. The control is to train **both** models and report the delta, which
turns an ambiguity into a measured quantity. `make_feature_frame` makes that one
argument, not a separate code path.

In [ ]:
spec_loans = cfg.dataset("loans")
X_full, y_loans = make_feature_frame(loans, spec_loans)
X_rate_free, _ = make_feature_frame(loans, spec_loans, extra_drop=["Interest_Rate"])

print("full model features     :", list(X_full.columns))
print()
print("rate-free model features:", list(X_rate_free.columns))
print()
print("LTV = Loan_Amount / Collateral_Value")
ltv = loans["Loan_Amount"] / loans["Collateral_Value"]
print(ltv.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(3).to_string())
print(f"\nShare of loans with LTV > 1 (under-collateralised): {(ltv > 1).mean():.1%}")
print("-> justifies the collateral-aware LGD sensitivity alongside the flat 40%.")

### 4.2 · M3 Fraud — `nova_transactions.csv`

In [ ]:
txn = frames["transactions"].copy()
txn["amount_vs_history"] = txn["Amount"] / txn["Historical_Avg_Transaction"]

display(class_balance(txn, "Fraud_Flag"))

for feature, log_scale in [("Amount", True), ("amount_vs_history", True)]:
    lift = decile_lift(txn, feature, "Fraud_Flag")
    display(lift[["bin", "n", "n_events", "event_rate", "lift"]])
    fig = viz.plot_decile_lift(lift, feature, "Fraud_Flag")
    save_figure(fig, f"01_m03_{feature}_lift", close=False); plt.show()

    fig = viz.plot_distribution_by_class(txn, feature, "Fraud_Flag", log_x=log_scale)
    save_figure(fig, f"01_m03_{feature}_distribution", close=False); plt.show()

In [ ]:
# Drift check: is the fraud rate stationary? This decides whether a
# time-based holdout is representative.
rates = temporal_rate(txn, "Timestamp", "Fraud_Flag", freq="ME")
fig = viz.plot_temporal_rate(rates, "Timestamp", "Fraud_Flag")
save_figure(fig, "01_m03_fraud_rate_over_time", close=False); plt.show()

print(f"monthly fraud rate: min {rates['event_rate'].min():.4f}, "
      f"max {rates['event_rate'].max():.4f}, overall {txn['Fraud_Flag'].mean():.4f}")

# Entity structure: devices are shared, which is why GroupKFold is a cross-check.
print(f"\ncustomers per Device_ID: mean "
      f"{txn.groupby('Device_ID')['Customer_ID'].nunique().mean():.1f}")
display(group_rate(txn, "Transaction_Type", "Fraud_Flag"))

### 4.3 · M4 Customer Analytics — the documented negative result

This is the module where the honest answer is *no model*. Two independent tests agree:
the strongest correlation with churn is ~0.02, and the per-fold AUC standard error at
89 positives is ~0.07 — a 95% interval of roughly ±0.14, wide enough that two genuinely
different models cannot be told apart.

A permutation test in Phase 4 will make the null explicit. The executive question —
*which 1,000 customers to contact* — is then answered on **value** (CLV × complaints ×
product depth), not on an unreliable churn score.

In [ ]:
customers = frames["customers"]
display(class_balance(customers, "Churn_Flag"))

numeric = customers.select_dtypes(include=[np.number]).drop(columns=["Customer_ID"])
churn_corr = numeric.drop(columns=["Churn_Flag"]).corrwith(
    customers["Churn_Flag"].astype("float64")
).sort_values(key=np.abs, ascending=False)
print("Correlation with Churn_Flag (all features):")
print(churn_corr.round(4).to_string())
print(f"\nSTRONGEST |correlation| = {churn_corr.abs().max():.4f}")

fig = viz.plot_target_correlations(churn_corr.head(12), "Churn_Flag")
save_figure(fig, "01_m04_churn_no_signal", close=False); plt.show()

In [ ]:
# CLV, by contrast, is highly predictable - because it is a DERIVED column.
# We therefore frame M4's regression as attribution, not forecasting (L-07).
clv_corr = numeric.drop(columns=["Estimated_CLV"]).corrwith(
    customers["Estimated_CLV"].astype("float64")
).sort_values(key=np.abs, ascending=False)
print("Correlation with Estimated_CLV:")
print(clv_corr.round(4).to_string())
print(f"\nrows with Estimated_CLV == 0: {(customers['Estimated_CLV'] == 0).sum()}")

### 4.4 · Cross-module integration check (N-02)

In [ ]:
overlap = check_entity_overlap(
    {k: frames[k] for k in ("loans", "customers", "transactions")}, "Customer_ID"
)
display(overlap)
print("loans and customers share ZERO customer ids -> no customer-level join exists.")
print("Module 11 therefore integrates at PORTFOLIO / SEGMENT level. This is a data")
print("constraint to state in the report, not something to engineer around.")

## 5 · Validation design — build it, then audit it

Eight modules, eight schemes. `make_splitter` reads the recipe from
`configs/config.yaml`, so the choice is configuration, never a hardcoded default.
Each scheme is then *audited*: the assertions below fail loudly if any training row
sits at or after a test row, or if an entity appears in two folds.

In [ ]:
from novafin.data.splits import assert_no_group_overlap, assert_no_temporal_overlap

market_frame = frames["market"]
splitter, kwargs = make_splitter("market", market_frame, cfg=cfg)
schedule = describe_splits(splitter, market_frame, **kwargs)
display(schedule)

for train_idx, test_idx in splitter.split(market_frame, **kwargs):
    assert_no_temporal_overlap(train_idx, test_idx, market_frame["Date"], embargo=5)
    assert_no_group_overlap(train_idx, test_idx, market_frame["Date"])
print("PASS - no training date at or after a test date; every date kept whole.")

fig = viz.plot_split_schedule(schedule, "purged walk-forward (embargo 5d)")
save_figure(fig, "01_m05_split_schedule", close=False); plt.show()

In [ ]:
hft_frame = frames["hft"]
splitter_hft, kwargs_hft = make_splitter("hft", hft_frame, cfg=cfg)
(train_idx, val_idx), = splitter_hft.split(hft_frame, **kwargs_hft)
test_idx = splitter_hft.test_index(hft_frame["Trading_Day"])

days = hft_frame["Trading_Day"]
print(f"train: {len(train_idx):>7,} rows over {days.iloc[train_idx].nunique()} days")
print(f"val  : {len(val_idx):>7,} rows over {days.iloc[val_idx].nunique()} days")
print(f"test : {len(test_idx):>7,} rows over {days.iloc[test_idx].nunique()} days")
assert_no_group_overlap(train_idx, val_idx, days)
assert_no_group_overlap(train_idx, test_idx, days)
print("PASS - whole trading days, never shuffled intraday.")

In [ ]:
train_idx, holdout_idx = holdout_by_time(txn, "transactions", cfg=cfg)
print(f"train  : {len(train_idx):,} rows  "
      f"{txn['Timestamp'].iloc[train_idx].min().date()} -> {txn['Timestamp'].iloc[train_idx].max().date()}  "
      f"fraud {txn['Fraud_Flag'].iloc[train_idx].mean():.4f}")
print(f"holdout: {len(holdout_idx):,} rows  "
      f"{txn['Timestamp'].iloc[holdout_idx].min().date()} -> {txn['Timestamp'].iloc[holdout_idx].max().date()}  "
      f"fraud {txn['Fraud_Flag'].iloc[holdout_idx].mean():.4f}")
print("\nThis block is scored ONCE, at the end, after every tuning decision.")

### 5.1 · How precise can each module's metric actually be?

Before modelling, it is worth knowing what precision the data can support. The
Hanley–McNeil closed form gives the standard error of a fold-level AUC from the number
of positives and negatives alone — no model required.

This is what justifies the repeated-CV choice quantitatively rather than by assertion.

In [ ]:
from novafin.data.validate import auc_standard_error

rows = []
for key in ("initiatives", "loans", "transactions", "customers"):
    spec = cfg.dataset(key)
    frame = frames[key]
    n_splits = int(cfg.splits(key).get("n_splits", 5))
    n_pos = int(frame[spec.target].sum())
    n_neg = len(frame) - n_pos
    se = auc_standard_error(n_pos / n_splits, n_neg / n_splits)
    rows.append({
        "module": key, "n_rows": len(frame), "n_positive": n_pos,
        "folds": n_splits, "pos_per_fold": round(n_pos / n_splits, 1),
        "AUC_std_error": round(se, 3), "95%_interval": f"+/-{1.96 * se:.3f}",
        "repeats_configured": cfg.splits(key).get("n_repeats", 1),
    })
pd.DataFrame(rows)

## 6 · Persist the EDA artefacts

Tables are written to `reports/tables/` so the D7 guide and D8 deck read them from disk
rather than re-deriving numbers. Figures are already in `reports/figures/`.

In [ ]:
cfg.paths.tables.mkdir(parents=True, exist_ok=True)

overview.to_csv(cfg.paths.tables / "01_dataset_overview.csv", index=False)
combined_report_frame(list(reports.values())).to_csv(
    cfg.paths.tables / "01_validation_findings.csv", index=False
)
overlap.to_csv(cfg.paths.tables / "01_entity_overlap.csv")
pd.DataFrame(rows).to_csv(cfg.paths.tables / "01_cv_power.csv", index=False)

for key, frame in frames.items():
    column_profile(frame).to_csv(cfg.paths.tables / f"01_profile_{key}.csv", index=False)

print("Written:")
for path in sorted(cfg.paths.tables.glob("01_*.csv")):
    print("  ", path.name)
print()
print("Figures:")
for path in sorted(cfg.paths.figures.glob("01_*.png")):
    print("  ", path.name)

---

## Phase 2 summary

| Guarantee | Established by |
|---|---|
| The data on disk **is** the audited data | schema assertion in `load_dataset` |
| Every leak is declared and removed | `validate_dataset` + `make_feature_frame` |
| The register is still accurate | `tests/test_leakage.py` (marked `needs_data`) |
| No split leaks | `assert_no_temporal_overlap` / `assert_no_group_overlap` |
| Metric precision is known before modelling | Hanley–McNeil standard errors above |

**NEXT:** `02_feature_engineering` — causal feature construction per module: the forward
equity target, trailing order-book features, customer/device aggregates computed with
`shift(1)` before `rolling`, option moneyness and the Black-Scholes residual.